# 07 Buffer Pool 与内存

## 八股问答

### 1. 什么是 Buffer Pool？

Buffer Pool 是 InnoDB 在内存中维护的一大块缓存，缓存数据页、索引页、undo 页、
自适应哈希索引、锁信息等。所有读写都先命中 Buffer Pool，没有才从磁盘加载，
所以它直接决定热点查询的速度。

### 2. 为什么要有 Buffer Pool？

磁盘随机读是微秒到毫秒级，内存访问是纳秒级，差距 3 到 4 个数量级。热点数据
缓存在内存里，绝大多数查询不需要磁盘 IO。InnoDB 通过 LRU 淘汰保持高命中率，
并通过延迟刷盘把随机写变成后台批量顺序写。

### 3. Buffer Pool 的结构？

按内存组织：

- Chunk：Buffer Pool 按 chunk（默认 128MB）划分，可在线调整大小。
- 控制块 + 缓存页：每页有控制块记录页号和状态，页默认 16KB。
- 三种链表：
  - Free List：空闲页，等待分配。
  - LRU List：已使用的数据页，按访问热度排序。
  - Flush List：脏页列表，按最早修改顺序等待刷盘。

按内容：数据页、索引页、undo 页、自适应哈希索引、锁信息等混放在同一片
Buffer Pool 中。

### 4. 如何管理 Buffer Pool？LRU 怎么优化？

标准 LRU 有两个问题：

- 全表扫描会瞬间把热点页挤出去（缓冲池污染）。
- 预读的冷页可能没被访问，白白占内存。

InnoDB 的优化：

1. 冷热分区 LRU：LRU 链表按大约 5:3 分成 young 区和 old 区。新读入的页先进
   old 区，只有再次访问才晋升 young 区；全表扫描的页很难污染热点。
2. 预读：顺序读时提前加载相邻页，并用 old 区缓冲，避免污染。
3. 脏页刷盘：后台线程按 Flush List 顺序刷盘，由
   `innodb_io_capacity` 控制刷盘速度；刷盘前会检查 redo 的 checkpoint。
4. Change Buffer：二级索引的插入、删除、更新不直接改索引页，先记录变更，
   等索引页被读到内存时再合并，减少随机 IO。

### 5. 内存相关关键参数？

| 参数 | 含义 | 常见设置 |
| --- | --- | --- |
| `innodb_buffer_pool_size` | Buffer Pool 总大小 | 物理内存 50%-70% |
| `innodb_buffer_pool_instances` | 分片数，减少锁竞争 | 按大小分多个实例 |
| `innodb_buffer_pool_chunk_size` | 最小扩容单位 | 默认 128MB |
| `innodb_old_blocks_time` | 冷页在 old 区最短停留时间 | 1000ms 左右 |
| `innodb_io_capacity` | 后台刷盘 IO 上限 | 按磁盘能力设置 |

查看命中率：`SHOW ENGINE INNODB STATUS` 中 Buffer pool hit rate。


## 面试话术

“Buffer Pool 是 InnoDB 性能的核心：读写先过内存，缺失才走磁盘。管理上靠
冷热分区 LRU 防止全表扫描污染热点，靠 Flush List 和后台线程控制刷盘速度，
靠 Change Buffer 延迟二级索引的随机写。排查性能时我会先看命中率、脏页数量和
redo 增长，判断是内存不足还是刷盘跟不上。”


## 自查清单

- [ ] 能说清 Buffer Pool 里存了哪几类内容
- [ ] 能画出 Free、LRU、Flush 三条链表的分工
- [ ] 能解释冷热分区 LRU 为什么能防全表扫描污染
- [ ] 能说清 Change Buffer 解决什么问题
- [ ] 能在终端读出命中率、脏页数和缓存页数
